In [19]:
import json
from pathlib import Path
import pandas as pd
import re
import extract as ext
import pdfplumber

In [20]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [21]:
with open(ROOT / "diccionarios_clasificacion_ejercicios/quimica/temas_ejercicio.json", "r") as file:
    temas_ejercicio = json.load(file)

with open(ROOT / "diccionarios_clasificacion_ejercicios/quimica/quimica_keywords.json") as file:
    quimica_keywords = json.load(file)

In [57]:
file = Path("enlace_químico/2022 - Enlace Químico.pdf")

In [66]:
def extract_statement(exercise):
    m = re.search("R E S O L U C I Ó N", exercise)
    if m:
        end_pos = m.start()
        exercise = exercise[:end_pos].strip()
        lines = exercise.split("\n")
        exam = lines[-1]
        statement = "\n".join(lines[:-1])
        return exam, statement
    return None

In [65]:
def extract_exercises(file):
    exercises = []
    with pdfplumber.open(file) as pdf:
        for idx, page in enumerate(pdf.pages):
            texto = ext.extract_text_from_page(page)
            if idx > 0:  # ejercicios comienzan en 2da página
                st = extract_statement(texto)
                if st:
                    exercises.append(extract_statement(texto))
    return exercises

In [67]:
exercises_list = []
for file in Path("enlace_químico").iterdir():   
    exams, statements = [], []
    print("Processing file: ", file.name)
    exercises = extract_exercises(file)
    for ex in exercises:
        if ex:
            exercises_list.append({"exam" : ex[0], "statement": ex[1]})

Processing file:  2026 - Enlace Químico.pdf
Processing file:  2014 - Enlace Químico.pdf
Processing file:  2002 - Enlace Químico.pdf
Processing file:  2008 - Enlace Químico.pdf
Processing file:  2024 - Enlace Químico.pdf
Processing file:  2016 - Enlace Químico.pdf
Processing file:  2000 - Enlace Químico.pdf
Processing file:  2010 - Enlace Químico.pdf
Processing file:  2006 - Enlace Químico.pdf
Processing file:  2022 - Enlace Químico.pdf
Processing file:  2018 - Enlace Químico.pdf
Processing file:  2012 - Enlace Químico.pdf
Processing file:  2004 - Enlace Químico.pdf
Processing file:  2020 - Enlace Químico.pdf
Processing file:  2003 - Enlace Químico.pdf
Processing file:  2015 - Enlace Químico.pdf
Processing file:  2001 - Enlace Químico.pdf
Processing file:  2017 - Enlace Químico.pdf
Processing file:  2009 - Enlace Químico.pdf
Processing file:  2025 - Enlace Químico.pdf
Processing file:  2023 - Enlace Químico.pdf
Processing file:  2019 - Enlace Químico.pdf
Processing

In [69]:
df = pd.DataFrame(exercises_list)

In [77]:
def classify_exercise(statement: str, topic: str, keywords = quimica_keywords) -> tuple:
    topic_keywords = keywords[topic]["keywords_tema"]

    exercise_topic = None
    for topic_kw in topic_keywords:
        if topic_kw in statement:
            exercise_topic = topic    
            break

    if exercise_topic:
        exercise_types = []
        exercise_type_keywords = keywords[topic]["tipos"]
        
        for tipo, tipo_keywords in exercise_type_keywords.items():
            if any(kw in statement for kw in tipo_keywords):
                exercise_types.append(tipo)
    else:
        exercise_topic, exercise_types = None, [] 

    return exercise_topic, exercise_types

In [71]:
df.head()

,exam,statement
0,QUÍMICA. 2026. JUNIO. EJERCICIO 1B,Justifique si son verdaderas o falsas las sigu...
1,QUÍMICA. 2026. RESERVA 1. EJERCICIO 1B,Dadas las moléculas C H C l y P F ₃ :\n3\na) R...
2,QUÍMICA. 2026. RESERVA 2. EJERCICIO 1B,Justifique si son verdaderas o falsas las sigu...
3,QUÍMICA. 2026. RESERVA 4. EJERCICIO 1B,"Justifique, atendiendo al tipo de enlace, las ..."
4,QUÍMICA. 2014. JUNIO. EJERCICIO 3. OPCIÓN B,a) Deduzca la geometría de las moléculas BCl y...


In [89]:
df["topic"], df["exercise_type"] = zip(*df.statement.apply(classify_exercise, topic="enlace_químico"))

In [91]:
df.isna().sum()

exam             0
statement        0
topic            1
exercise_type    0
dtype: int64